# Buckling of a laminated cylindrical panel in axial compression: Jaunky and Knight (1999)

The half-cylinder panel of Jaunky and Knight [1] (Section 3.2, Table 2, Fig. 3) with the laminate
$[\pm 70/\pm 70/70]_s$ under uniform axial compression, with the models `'cylshell_clpt_donnell'` and
`'cylshell_clpt_sanders'`, against their Donnell and Sanders-Koiter buckling loads of Table 2 ($R/t = 600$).

**Panel.** Length $L = 60$ in, radius $R = 6$ in, an arc of $\pi$ radians ($b = \pi R$) and thickness $t = 0.01$
in; the laminate $[\pm 70/\pm 70/70]_s$ has ten plies of equal thickness with $E_{11} = 13.75$ Msi,
$E_{22} = 1.03$ Msi, $G_{12} = G_{13} = G_{23} = 0.42$ Msi and $\nu_{12} = 0.25$, the angles measured from the axis.
Boundary conditions 1 of [1]: $u = v = w = 0$ at $x = 0$, $v = w = 0$ at $x = L$, $u = w = 0$ on the straight
edges, the rotations free; in panels `x2u = 1` and `y1v = y2v = 1`. The prebuckling state is a uniform $N_{xx}$, as
in [1]. Units: inches and psi, the buckling load in lb/in.

Table 2 of [1] gives 31.91 lb/in with Donnell's and 30.79 lb/in with Sanders-Koiter's theory.

In [1]:
import time

import numpy as np
from IPython.display import Markdown, display
from structsolve import lb

from panels.shell import Shell

REF = {'cylshell_clpt_donnell': 31.91, 'cylshell_clpt_sanders': 30.79}  # lb/in, Table 2 of [1]


def panel(model, m):
    s = Shell()
    s.m = m
    s.n = m
    s.stack = [70, -70, 70, -70, 70, 70, -70, 70, -70, 70]
    s.plyt = 0.01/len(s.stack)
    s.laminaprop = (13.75e6, 1.03e6, 0.25, 0.42e6, 0.42e6, 0.42e6)
    s.model = model
    R = 6.
    s.a = 60.
    s.b = np.pi*R
    s.r = R
    # u free at x = a, v free on the straight edges
    s.x2u = 1
    s.y1v = 1
    s.y2v = 1
    s.Nxx = -1
    return s

## Buckling loads and convergence

First load multiplier with $m = n$ Bardell terms, up to 30, the largest number of terms of panels; the
difference to Table 2 of [1] in parentheses.

In [2]:
rows = []
t0 = time.time()
for m in (18, 24, 30):
    row = [m]
    for model, ref in REF.items():
        s = panel(model, m)
        eigvals, eigvecs = lb(s.calc_kC(), s.calc_kG(), silent=True)
        Ncr = float(np.real(eigvals[0]))
        row.append('%.2f (%+.2f%%)' % (Ncr, 100*(Ncr/ref - 1)))
    rows.append(row)
print('%.0f s' % (time.time() - t0))
lines = ['| $m = n$ | Donnell, lb/in (vs 31.91) | Sanders, lb/in (vs 30.79) |', '|---|---|---|']
lines += ['| %d | %s | %s |' % tuple(r) for r in rows]
display(Markdown('\n'.join(lines)))

47 s


| $m = n$ | Donnell, lb/in (vs 31.91) | Sanders, lb/in (vs 30.79) |
|---|---|---|
| 18 | 32.02 (+0.34%) | 31.35 (+1.81%) |
| 24 | 31.78 (-0.41%) | 31.26 (+1.53%) |
| 30 | 31.73 (-0.56%) | 31.25 (+1.50%) |

## Discussion

- With $m = n = 30$ the Donnell and Sanders models of panels are within 2 % of the Donnell and Sanders-Koiter
  values of Table 2 of [1].
- Only this row of Tables 2 and 3 of [1] is used. The other rows are not reproduced within 2 % with the same model
  and boundary conditions ($m = n = 30$): $R = 6$ in, $t = 0.06$ in: Donnell +7.7 %, Sanders-Koiter +20 %
  (isotropic: +2.3 %, +5.4 %); $R = 24$ in, $t = 0.24$ in: -3.1 %, -0.5 % (isotropic: +1.0 %, +0.6 %); $R = 48$ in,
  $t = 0.24$ in: -7.9 % both. For isotropic panels the Donnell model of panels reproduces the exact solution of
  Batdorf [2] to $10^{-12}$, so the deviations come either from the solutions of [1] or from the reading of their
  boundary conditions.

## References

1. N. Jaunky and N. F. Knight, Jr., "An assessment of shell theories for buckling of circular cylindrical laminated
   composite panels loaded in axial compression", International Journal of Solids and Structures, 36, 3799-3820,
   1999. https://doi.org/10.1016/S0020-7683(98)00177-2
2. S. B. Batdorf, "A Simplified Method of Elastic-Stability Analysis for Thin Cylindrical Shells", NACA Report 874,
   National Advisory Committee for Aeronautics, 1947. https://ntrs.nasa.gov/citations/19930091945